# Crimes Database

### TOC

### Introduction

In this project, guided by DataQuest, we will explore the core principles of database administration and design using **PostgreSQL**. Our objective is to transition from raw data to a structured, secure, and optimized relational database environment. 

The workflow begins with the creation of a custom schema and a primary table specifically tailored to our dataset. After populating the database via a `data.csv` import, we will conduct a detailed analysis of data types to ensure storage efficiency. To conclude, we will implement a professional security model by defining distinct user groups and assigning role-based permissions. This project demonstrates not only how to store data, but how to manage it with an emphasis on integrity, performance, and the Principle of Least Privilege.

**Rough Project Overview:**

![project overview](project-overview.png)

### Setup

To begin, we will establish a connection using the specific Postgres user created for this project. Once connected, we will execute the commands necessary to initialize our database and schema.

In [1]:
import psycopg2

conn = psycopg2.connect(
    dbname="dq", 
    user="dq",         # user created for project purposes only
    password="abc123",
    host="localhost"
)
conn.autocommit = True
cur = conn.cursor()
cur.execute("CREATE DATABASE crimes_db;")
conn.close()

In [2]:
conn = psycopg2.connect(
    dbname="crimes_db", 
    user="dq",         # user created for project purposes only
    password="abc123",
    host="localhost"
)
conn.autocommit = True
cur = conn.cursor()
cur.execute("CREATE SCHEMA crimes;")

### Data

With the database and schema prepared, we are now ready to process our information. We will begin by using the `csv` module to import and read the raw data.

In [3]:
import csv
with open('boston.csv') as file:
    reader = csv.reader(file)
    col_headers = next(reader)
    first_row = next(reader)

print(col_headers)
print(first_row)

['incident_number', 'offense_code', 'description', 'date', 'day_of_the_week', 'lat', 'long']
['1', '619', 'LARCENY ALL OTHERS', '2018-09-02', 'Sunday', '42.35779134', '-71.13937053']


### Data Types

Before migrating the data into our schema’s table, we will analyze the data types of each column to ensure the database is as optimized as possible. To facilitate this, we will define a custom function that compiles a set of all unique values within a specific column. By examining these unique values, we can accurately determine the most appropriate data type for every field.

In [4]:
def get_col_set(csv_file, col_index):
    values = set()
    with open(csv_file, 'r') as f:
        next(f)
        reader = csv.reader(f)
        for row in reader:
            values.add(row[col_index])
    return values

for i in range(len(col_headers)):
    values = get_col_set("boston.csv", i)
    print(col_headers[i], len(values), sep='\t')

incident_number	298329
offense_code	219
description	239
date	1177
day_of_the_week	7
lat	18177
long	18177


Having identified the number of unique values in each column, we will now focus on the two columns containing textual data: `description` and `day_of_the_week`. Our objective is to identify the longest possible string in these columns to determine the most efficient data types. 

The `day_of_the_week` column contains only seven unique values, representing the days of the week; therefore, "Wednesday" is the longest possible string. The `description` column, however, contains 239 unique values. To find the maximum string length for this column, we will use our custom `get_col_set()` function to generate a set of all unique strings and then compute the length of the longest entry.

In [5]:
descriptions = get_col_set("boston.csv", 2)
longest_string = max(descriptions, key=len) 
print(longest_string)
print(len(longest_string))

RECOVERED - MV RECOVERED IN BOSTON (STOLEN OUTSIDE BOSTON)
58


### Column Data Type Selection

Based on the analysis above, we can now assign the most appropriate data types to our table columns. As a reminder, our data includes the following fields:

* `incident_number`
* `offense_code`
* `description`
* `date`
* `day_of_the_week`
* `lat`
* `long`

We will assign the `INTEGER` type to `incident_number` and `offense_code` since they contain numeric data. For the `description` column, we will use `VARCHAR(100)`, which provides ample space for the longest strings while remaining memory-efficient. The `date` column will use the `DATE` type, while `lat` and `long` will be stored as `DECIMAL`. Finally, because the `day_of_the_week` column contains a small, fixed set of values, we will use an `ENUM` type for better organization and performance.

**Summary of Data Type Assignments:**

| Column Name | Data Type |
| :--- | :--- |
| `incident_number` | INTEGER |
| `offense_code` | INTEGER |
| `description` | VARCHAR(100) |
| `date` | DATE |
| `day_of_the_week` | ENUM |
| `lat` | DECIMAL |
| `long` | DECIMAL |

We will proceed by creating the table and defining these data types in the code block below.

In [6]:
# Create the enumerated datatype for representing the weekday
cur.execute("""
    CREATE TYPE weekday AS ENUM ('Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday');
""")

# Create the table
cur.execute("""
    CREATE TABLE crimes.boston_crimes (
        incident_number INTEGER PRIMARY KEY,
        offense_code INTEGER,
        description VARCHAR(100),
        date DATE,
        day_of_the_week weekday,
        lat decimal,
        long decimal
    );
""")

### Data Migration

With the table structure finalized within our schema, we are now ready to load the data.

In [7]:
# loading the data into our boston_crimes table
with open("boston.csv") as f:
    cur.copy_expert("COPY crimes.boston_crimes FROM STDIN WITH CSV HEADER;", f)

# ensuring the data was loaded successfully by printing the number of rows
cur.execute("SELECT * FROM crimes.boston_crimes")
print(len(cur.fetchall()))

298329


### Users and Permissions

Now that the database, schema, and table are fully established, we will proceed to create our user groups and users. We intend to implement the **Principle of Least Privilege** by creating two distinct groups with specific permissions. Once these groups are established, we will create and assign one user to each.

To begin this process, we will revoke all `PUBLIC` privileges from the database. This ensures that no users possess unintended permissions by default. After securing the environment, we will explicitly grant the necessary permissions to the appropriate groups.

In [9]:
cur.execute("REVOKE ALL ON SCHEMA public FROM public;")
cur.execute("REVOKE ALL ON DATABASE crimes_db FROM public;")

With the database secured, we will create our first user group. This group will be granted **read-only** access, which is appropriate for roles such as a Data Analyst.

In [11]:
cur.execute("CREATE GROUP readonly NOLOGIN;")
cur.execute("GRANT CONNECT ON DATABASE crimes_db TO readonly;")
cur.execute("GRANT USAGE ON SCHEMA crimes TO readonly;")
cur.execute("GRANT SELECT ON ALL TABLES IN SCHEMA crimes TO readonly;")

Our second user group will be granted both **read and write** permissions, suitable for Data Science roles or users requiring more administrative flexibility.

In [12]:
cur.execute("CREATE GROUP readwrite NOLOGIN;")
cur.execute("GRANT CONNECT ON DATABASE crimes_db TO readwrite;")
cur.execute("GRANT USAGE ON SCHEMA crimes TO readwrite;")
cur.execute("GRANT SELECT, INSERT, DELETE, UPDATE ON ALL TABLES IN SCHEMA crimes TO readwrite;")

Having established the groups and their respective privileges, we will now create a specific user for each. We will assign the `data_analyst` user to the read-only group and the `data_scientist` user to the read-write group.

In [13]:
cur.execute("CREATE USER data_analyst WITH PASSWORD 'secret1';")
cur.execute("GRANT readonly TO data_analyst;")

cur.execute("CREATE USER data_scientist WITH PASSWORD 'secret2';")
cur.execute("GRANT readwrite TO data_scientist;")

### Verification

In this final section, we will verify that our user groups and permissions were configured successfully. We will confirm that the groups are assigned the correct privileges by querying the `information_schema.table_privileges` table.

In [14]:
# closing the old connection to test our users and groups on new connection
conn.close()

In [23]:
# new connection
conn = psycopg2.connect(
    dbname="crimes_db", 
    user="dq",         # user created for project purposes only
    password="abc123",
    host="localhost"
)
conn.autocommit = True
cur = conn.cursor()

In [24]:
# custom function for printing and retaining SQL queries
def sql_queries(cur):
    users = cur.fetchall()
    for user in users:
        print(user)
    return users

In [25]:
cur.execute("""
    SELECT grantee, privilege_type
    FROM information_schema.table_privileges
    WHERE grantee = 'readonly';
""")

analyist_privilages = sql_queries(cur)

('readonly', 'SELECT')


In [26]:
cur.execute("""
    SELECT grantee, privilege_type
    FROM information_schema.table_privileges
    WHERE grantee = 'readwrite';
""")

scientist_privilages = sql_queries(cur)

('readwrite', 'INSERT')
('readwrite', 'SELECT')
('readwrite', 'UPDATE')
('readwrite', 'DELETE')


In [27]:
# closing the final connection
conn.close()

The results above confirm that our groups and users have been granted the appropriate privileges and are functioning as intended.

### Conclusion

In this project, we successfully demonstrated the end-to-end workflow of initializing and securing a relational database. By structuring the **`boston_crimes`** table within a dedicated schema and database, we established a clean, organized hierarchy for raw data. Beyond simple data migration, this project highlighted the critical role of data validation; by analyzing unique values and string lengths, we ensured that our data types were optimized for both performance and storage efficiency.

Furthermore, we moved beyond basic functionality to implement a robust security model. By applying the **Principle of Least Privilege**, we demonstrated how to protect sensitive information through role-based access control. Creating distinct user groups for Data Analysts and Data Scientists reflects real-world best practices, ensuring that users have exactly the permissions they need—and nothing more. This project serves as a comprehensive foundation for building scalable, secure, and well-optimized database systems.